# MVTec AD — Unsupervised Anomaly Detection for Manufacturing Defect Inspection

**Approach:** Train a ResNet18-based Autoencoder on normal samples only. At inference, anomalous regions produce high reconstruction error.

**Evaluation:** Image-level AUROC + Pixel-level AUROC (with Youden's J for optimal threshold)

**Dataset:** [MVTec Anomaly Detection Dataset](https://www.mvtec.com/company/research/datasets/mvtec-ad) — 15 categories, 5000+ images

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | PyTorch: {torch.__version__} | NumPy: {np.__version__}")

# ── Data path (adjust if your dataset name differs) ─────────────────────────
# Kaggle datasets are mounted at /kaggle/input/{dataset-name}/
DATA_ROOT = Path("/kaggle/input/mvtec-ad")
# If you uploaded the tar.xz, it may need extracting first:
if not (DATA_ROOT / "bottle").exists():
    # Try common alternative paths
    for candidate in [
        Path("/kaggle/input/mvtec-anomaly-detection"),
        Path("/kaggle/input/mvtec-ad/mvtec_anomaly_detection"),
    ]:
        if (candidate / "bottle").exists():
            DATA_ROOT = candidate
            break
    else:
        print("⚠️ Dataset not found! Check your Kaggle input path:")
        os.system("find /kaggle/input -maxdepth 3 -type d | head -30")

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Categories: {sorted([d for d in os.listdir(DATA_ROOT) if (DATA_ROOT/d).is_dir()])}")

## 1. Dataset & Model Definition

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────

class MVTecDataset(Dataset):
    def __init__(self, data_root, category, split="train", size=256, transform=None):
        self.size = size
        self.transform = transform or transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.images, self.labels, self.mask_paths = [], [], []
        cat_dir = Path(data_root) / category

        if split == "train":
            good_dir = cat_dir / "train" / "good"
            for p in sorted(good_dir.glob("*.png")):
                self.images.append(str(p)); self.labels.append(0); self.mask_paths.append(None)
        else:
            for defect in sorted(os.listdir(cat_dir / "test")):
                d = cat_dir / "test" / defect
                if not d.is_dir(): continue
                is_good = defect == "good"
                for p in sorted(d.glob("*.png")):
                    self.images.append(str(p))
                    self.labels.append(0 if is_good else 1)
                    if is_good:
                        self.mask_paths.append(None)
                    else:
                        self.mask_paths.append(str(
                            cat_dir / "ground_truth" / defect / p.name.replace(".png", "_mask.png")
                        ))

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx])
        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA)
        img = cv2.GaussianBlur(img, (3, 3), 0)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img_tensor = self.transform(img)

        if self.mask_paths[idx]:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
            mask_tensor = torch.tensor(mask / 255.0, dtype=torch.float32)
        else:
            mask_tensor = torch.zeros(self.size, self.size, dtype=torch.float32)

        return img_tensor, self.labels[idx], mask_tensor

# ── Model ────────────────────────────────────────────────────────────────────

class AnomalyAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.encoder = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4,
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

print("Dataset & Model defined.")

## 2. Train on 3 Categories (bottle, carpet, hazelnut)

In [ ]:
CATEGORIES = ["bottle", "carpet", "hazelnut"]
EPOCHS = 100
BATCH_SIZE = 16
LR = 1e-3
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

def train_category(category):
    print(f"\n{'='*60}")
    print(f"Training: {category}")
    print(f"{'='*60}")

    train_ds = MVTecDataset(DATA_ROOT, category, split="train")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    print(f"Training on {len(train_ds)} normal images")

    model = AnomalyAutoencoder().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.MSELoss()

    best_loss = float("inf")
    losses = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for imgs, _, _ in train_loader:
            imgs = imgs.to(device)
            recon = model(imgs)
            loss = criterion(recon, imgs)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        scheduler.step()

        if epoch % 20 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.6f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "loss": best_loss, "category": category,
            }, f"{SAVE_DIR}/{category}_best.pt")

    print(f"  Best loss: {best_loss:.6f}")
    return model, losses

# Train all 3 categories
all_models = {}
all_losses = {}
for cat in CATEGORIES:
    model, losses = train_category(cat)
    all_models[cat] = model
    all_losses[cat] = losses

print("\n All training complete!")

## 3. Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cat in CATEGORIES:
    ax.plot(all_losses[cat], label=cat, linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss"); ax.set_title("Training Loss (Normal Images Only)")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 4. Evaluate: Image-level & Pixel-level AUROC

In [ ]:
def evaluate_category(model, category):
    model.eval()
    test_ds = MVTecDataset(DATA_ROOT, category, split="test")
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

    anomaly_maps, labels, gt_masks = [], [], []
    with torch.no_grad():
        for imgs, lbls, masks in test_loader:
            recon = model(imgs.to(device))
            error = ((imgs.to(device) - recon) ** 2).mean(dim=1).cpu().numpy()
            for i in range(len(lbls)):
                anomaly_maps.append(error[i])
                labels.append(lbls[i].item())
                gt_masks.append(masks[i].numpy())

    labels = np.array(labels)
    image_scores = np.array([m.max() for m in anomaly_maps])

    # Image-level
    img_auroc = roc_auc_score(labels, image_scores)
    fpr, tpr, thresholds = roc_curve(labels, image_scores)
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    img_f1 = f1_score(labels, (image_scores >= best_thresh).astype(int))

    # Pixel-level
    px_scores = np.concatenate([m.flatten() for m, g in zip(anomaly_maps, gt_masks) if g.max() > 0])
    px_labels = np.concatenate([(g > 0.5).astype(int).flatten() for g in gt_masks if g.max() > 0])
    pix_auroc = roc_auc_score(px_labels, px_scores)

    return {
        "category": category,
        "image_auroc": img_auroc, "image_f1": img_f1,
        "pixel_auroc": pix_auroc, "threshold": best_thresh,
        "anomaly_maps": anomaly_maps, "labels": labels, "gt_masks": gt_masks,
        "fpr": fpr, "tpr": tpr,
    }

# Evaluate all categories
results = {}
for cat in CATEGORIES:
    r = evaluate_category(all_models[cat], cat)
    results[cat] = r
    print(f"{cat:12s} | Image AUROC: {r['image_auroc']:.4f} | Image F1: {r['image_f1']:.4f} | Pixel AUROC: {r['pixel_auroc']:.4f}")

print("\n" + "="*70)
avg_img = np.mean([r["image_auroc"] for r in results.values()])
avg_pix = np.mean([r["pixel_auroc"] for r in results.values()])
print(f"{'AVERAGE':12s} | Image AUROC: {avg_img:.4f} | Pixel AUROC: {avg_pix:.4f}")

## 5. ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, cat in enumerate(CATEGORIES):
    r = results[cat]
    axes[i].plot(r["fpr"], r["tpr"], linewidth=2, color="#E74C3C")
    axes[i].plot([0,1], [0,1], "k--", alpha=0.3)
    axes[i].fill_between(r["fpr"], r["tpr"], alpha=0.15, color="#E74C3C")
    axes[i].set_title(f"{cat} (AUROC={r['image_auroc']:.3f})", fontsize=13, fontweight="bold")
    axes[i].set_xlabel("False Positive Rate"); axes[i].set_ylabel("True Positive Rate")
    axes[i].grid(True, alpha=0.3)
fig.suptitle("Image-level ROC Curves", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Anomaly Heatmap Visualization

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for cat in CATEGORIES:
    r = results[cat]
    model = all_models[cat]
    model.eval()
    test_ds = MVTecDataset(DATA_ROOT, cat, split="test")

    # Pick 4 anomaly + 2 normal samples
    anom_idx = [i for i in range(len(test_ds)) if test_ds.labels[i] == 1][:4]
    norm_idx = [i for i in range(len(test_ds)) if test_ds.labels[i] == 0][:2]
    indices = anom_idx + norm_idx

    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    fig.suptitle(f"MVTec AD — {cat}", fontsize=16, fontweight="bold")

    for row, idx in enumerate(indices):
        img_t, label, gt_mask = test_ds[idx]
        with torch.no_grad():
            recon = model(img_t.unsqueeze(0).to(device)).cpu().squeeze(0)
        error = ((img_t - recon) ** 2).mean(dim=0).numpy()

        img_disp = np.clip(img_t.permute(1,2,0).numpy() * std + mean, 0, 1)
        recon_disp = np.clip(recon.permute(1,2,0).numpy() * std + mean, 0, 1)

        tag = "ANOMALY" if label == 1 else "NORMAL"
        axes[row,0].imshow(img_disp); axes[row,0].set_title(f"Original ({tag})"); axes[row,0].axis("off")
        axes[row,1].imshow(recon_disp); axes[row,1].set_title("Reconstruction"); axes[row,1].axis("off")
        axes[row,2].imshow(img_disp); axes[row,2].imshow(error, cmap="jet", alpha=0.5, vmin=0, vmax=np.percentile(error, 99))
        axes[row,2].set_title(f"Anomaly Map (max={error.max():.4f})"); axes[row,2].axis("off")
        axes[row,3].imshow(gt_mask.numpy(), cmap="gray", vmin=0, vmax=1); axes[row,3].set_title("Ground Truth"); axes[row,3].axis("off")

    plt.tight_layout(); plt.show()

## 7. Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Category": r["category"],
        "Image AUROC": f"{r['image_auroc']:.4f}",
        "Image F1": f"{r['image_f1']:.4f}",
        "Pixel AUROC": f"{r['pixel_auroc']:.4f}",
        "Threshold (Youden's J)": f"{r['threshold']:.6f}",
    }
    for r in results.values()
])
display(summary)

print("\nMethod: ResNet18 Autoencoder (unsupervised)")
print("Training: Normal images only | Loss: MSE | Optimizer: Adam + CosineAnnealing")
print("Anomaly score: max pixel-wise reconstruction error")
print("Threshold selection: Youden's J statistic (TPR - FPR maximization)")